# 使用 Milvus 和 DeepSeek 构建民法典知识库 RAG

DeepSeek 帮助开发者使用高性能语言模型构建和扩展 AI 应用。它提供高效的推理、灵活的 API 以及先进的专家混合 (MoE) 架构，用于强大的推理和检索任务。

在本教程中，我们将展示如何使用 Milvus 和 DeepSeek 构建一个基于《中华人民共和国民法典》的检索增强生成 (RAG) 管道。

## 准备工作

### 依赖与环境

In [1]:
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

Looking in indexes: http://mirrors.tencentyun.com/pypi/simple


---

In [ ]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

### 准备数据

我们使用《中华人民共和国民法典》（物权编 + 合同编）作为 RAG 的私有知识库，这是一个法律领域 RAG 管道的良好数据源。

将民法典文档 `mfd.md` 放置在 notebook 同级目录下。

文档路径：`C:\Users\67694\Desktop\新的开始\全栈\04\deepseek-quickstart-main\deepseek-quickstart-main\deepseek\api\mfd.md`

In [3]:
# 将 mfd.md 复制到当前工作目录（如已有可跳过）
# import shutil
# shutil.copy(r"C:\Users\67694\Desktop\新的开始\全栈\04\deepseek-quickstart-main\deepseek-quickstart-main\deepseek\api\mfd.md", "./mfd.md")

我们从 `mfd.md` 文件加载民法典内容。由于法律条文以 `**第X条**` 格式标记，我们使用正则表达式按法条进行分割，每条法律条文作为一个独立的文本块，这样能更精确地进行语义检索。

In [3]:
import re

# 读取民法典文档
mfd_path = r"mfd.md"
with open(mfd_path, "r", encoding="utf-8") as f:
    mfd_text = f.read()

# 按法条分割：以 **第XXX条** 为分割点，每条法律条文作为一个文本块
# 正则匹配 **第二百零四条**、**第三百九十九条** 等格式
pattern = r"(?=\*\*第[一二三四五六七八九十百千零〇]+条\*\*)"
raw_chunks = re.split(pattern, mfd_text)

# 过滤空块，并清理文本
text_lines = []
for chunk in raw_chunks:
    chunk = chunk.strip()
    if chunk and '条' in chunk[:20]:  # 确保是以法条开头的内容
        text_lines.append(chunk)

print(f"共分割出 {len(text_lines)} 条法律条文")
print(f"\n示例（前3条）：")
for i, line in enumerate(text_lines[:3]):
    print(f"\n--- 第{i+1}条 ---")
    print(line[:200])

共分割出 387 条法律条文

示例（前3条）：

--- 第1条 ---
**第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。

--- 第2条 ---
**第二百零五条** 本编调整因物的归属和利用产生的民事关系。

--- 第3条 ---
**第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。
国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。
国家实行社会主义市场经济，保障一切市场主体的平等法律地位和发展权利。


In [4]:
len(text_lines)

387

### 准备 LLM 和 Embedding 模型

DeepSeek 支持 OpenAI 风格的 API，您可以使用相同的 API 进行微小调整来调用 LLM。

In [5]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

定义一个 embedding 模型，使用 `milvus_model` 来生成文本嵌入。我们以 `DefaultEmbeddingFunction` 模型为例，这是一个预训练的轻量级嵌入模型。

In [ ]:
# from pymilvus import model as milvus_model

# embedding_model = milvus_model.DefaultEmbeddingFunction()

from pymilvus import model as milvus_model

# OpenAI国内代理 https://api.apiyi.com/token 
embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-3-large', # Specify the model name
    api_key='sk-***', # Provide your OpenAI API key
    base_url='https://api.apiyi.com/v1',
    dimensions=512
)

/home/ubuntu/miniconda3/envs/deepseek/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


生成一个测试嵌入并打印其维度和前几个元素。

In [7]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

512
[-0.02815247  0.00431061 -0.01856995  0.08197021 -0.03164673 -0.05267334
 -0.04876709  0.12487793 -0.02079773  0.0397644 ]


In [8]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[-0.00570679  0.02241516 -0.01895142  0.12792969 -0.01242828 -0.07318115
 -0.00282288  0.08618164 -0.04367065  0.03074646]


## 将数据加载到 Milvus

### 创建 Collection

In [9]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./milvus_mfd.db")

collection_name = "mfd_rag_collection"

关于 `MilvusClient` 的参数：

*   将 `uri` 设置为本地文件，例如 `./milvus_mfd.db`，是最方便的方法，因为它会自动利用 Milvus Lite 将所有数据存储在此文件中。
*   如果您有大规模数据，可以在 Docker 或 Kubernetes 上设置性能更高的 Milvus 服务器。在此设置中，请使用服务器 URI，例如 `http://localhost:19530`，作为您的 `uri`。
*   如果您想使用 Zilliz Cloud（Milvus 的完全托管云服务），请调整 `uri` 和 `token`，它们对应 Zilliz Cloud 中的 Public Endpoint 和 Api key。

检查 collection 是否已存在，如果存在则删除它。

In [10]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

创建一个具有指定参数的新 collection。

如果我们不指定任何字段信息，Milvus 将自动创建一个默认的 `id` 字段作为主键，以及一个 `vector` 字段来存储向量数据。一个保留的 JSON 字段用于存储非 schema 定义的字段及其值。

`metric_type` (距离度量类型):
     作用：定义如何计算向量之间的相似程度。
     例如：`IP` (内积) - 值越大通常越相似；`L2` (欧氏距离) - 值越小越相似；`COSINE` (余弦相似度) - 通常转换为距离，值越小越相似。
     选择依据：根据你的嵌入模型的特性和期望的相似性定义来选择。

 `consistency_level` (一致性级别):
     作用：定义数据写入后，读取操作能多快看到这些新数据。
     例如：
         `Strong` (强一致性): 总是读到最新数据，可能稍慢。
         `Bounded` (有界过期): 可能读到几秒内旧数据，性能较好 (默认)。
         `Session` (会话一致性): 自己写入的自己能立刻读到。
         `Eventually` (最终一致性): 最终会读到新数据，但没时间保证，性能最好。
     选择依据：在数据实时性要求和系统性能之间做权衡。

简单来说：
 `metric_type`：怎么算相似。
 `consistency_level`：新数据多久能被读到。

In [11]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [12]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|███████████████████████████████████| 387/387 [00:00<00:00, 1241924.75it/s]


{'insert_count': 387, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 

## 构建 RAG

### 检索查询数据

我们向民法典知识库提出 2 个问题进行检索和回答。

In [13]:
questions = [
    "不动产物权未经登记是否发生效力？",
    "抵押权人能否未经抵押人同意转让抵押财产？",
]

对每个问题在 collection 中搜索，并检索语义上最匹配的前3个结果。

In [14]:
all_search_res = []
for q in questions:
    res = milvus_client.search(
        collection_name=collection_name,
        data=embedding_model.encode_queries(
            [q]
        ),  # 将问题转换为嵌入向量
        limit=3,  # 返回前3个结果
        search_params={"metric_type": "IP", "params": {}},  # 内积距离
        output_fields=["text"],  # 返回 text 字段
    )
    all_search_res.append(res)

让我们看一下每个问题的搜索结果

In [15]:
import json

for i, (q, search_res) in enumerate(zip(questions, all_search_res)):
    print(f"\n===== 问题 {i+1}：{q} =====")
    retrieved_lines_with_distances = [
        (res["entity"]["text"], res["distance"]) for res in search_res[0]
    ]
    print(json.dumps(retrieved_lines_with_distances, indent=4, ensure_ascii=False))


===== 问题 1：不动产物权未经登记是否发生效力？ =====
[
    [
        "**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。\n依法属于国家所有的自然资源，所有权可以不登记。",
        0.6725884675979614
    ],
    [
        "**第四百三十六条** 设立质权，当事人应当依照法律规定办理登记。\n未经登记，不发生物权效力。",
        0.6193746328353882
    ],
    [
        "**第二百一十四条** 不动产物权的设立、变更、转让和消灭，依照法律规定应当登记的，自记载于不动产登记簿时发生效力。",
        0.6159839630126953
    ]
]

===== 问题 2：抵押权人能否未经抵押人同意转让抵押财产？ =====
[
    [
        "**第四百一十六条** 抵押期间，抵押人未经抵押权人同意，不得转让抵押财产，但是受让人代为清偿债务或者向抵押权人提供担保的除外。\n抵押人转让抵押财产的，转让所得的价款，应当向抵押权人提前清偿债务或者提存。",
        0.6951682567596436
    ],
    [
        "**第四百四十条** 质权期间，质权人未经出质人同意，不得转让质押财产，但是受让人代为清偿债务或者向质权人提供担保的除外。\n质权人转让质押财产的，转让所得的价款，应当向质权人提前清偿债务或者提存。",
        0.6561305522918701
    ],
    [
        "**第四百一十九条** 抵押权人实现抵押权，可以与抵押人协议以抵押财产折价或者以拍卖、变卖该抵押财产所得的价款优先受偿。\n协议损害其他债权人利益的，其他债权人可以请求人民法院撤销该协议。\n抵押权人与抵押人未就抵押权的实现方式达成协议的，抵押权人可以请求人民法院拍卖、变卖抵押财产。",
        0.6355683207511902
    ]
]


### 使用 LLM 获取 RAG 响应

将每个问题检索到的法律条文转换为字符串格式。

In [16]:
all_contexts = []
for i, search_res in enumerate(all_search_res):
    retrieved_lines_with_distances = [
        (res["entity"]["text"], res["distance"]) for res in search_res[0]
    ]
    context = "\n".join(
        [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
    )
    all_contexts.append(context)
    print(f"问题 {i+1} 检索到 {len(retrieved_lines_with_distances)} 条相关法条")

问题 1 检索到 3 条相关法条
问题 2 检索到 3 条相关法条


In [17]:
for i, (q, ctx) in enumerate(zip(questions, all_contexts)):
    print(f"\n===== 问题 {i+1}：{q} =====")
    print(ctx[:300] + "...")


===== 问题 1：不动产物权未经登记是否发生效力？ =====
**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。
依法属于国家所有的自然资源，所有权可以不登记。
**第四百三十六条** 设立质权，当事人应当依照法律规定办理登记。
未经登记，不发生物权效力。
**第二百一十四条** 不动产物权的设立、变更、转让和消灭，依照法律规定应当登记的，自记载于不动产登记簿时发生效力。...

===== 问题 2：抵押权人能否未经抵押人同意转让抵押财产？ =====
**第四百一十六条** 抵押期间，抵押人未经抵押权人同意，不得转让抵押财产，但是受让人代为清偿债务或者向抵押权人提供担保的除外。
抵押人转让抵押财产的，转让所得的价款，应当向抵押权人提前清偿债务或者提存。
**第四百四十条** 质权期间，质权人未经出质人同意，不得转让质押财产，但是受让人代为清偿债务或者向质权人提供担保的除外。
质权人转让质押财产的，转让所得的价款，应当向质权人提前清偿债务或者提存。
**第四百一十九条** 抵押权人实现抵押权，可以与抵押人协议以抵押财产折价或者以拍卖、变卖该抵押财产所得的价款优先受偿。
协议损害其他债权人利益的，其他债权人可以请求人民法院撤销该协议。
抵押权人...


In [18]:
questions

['不动产物权未经登记是否发生效力？', '抵押权人能否未经抵押人同意转让抵押财产？']

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的法律条文组装而成的。

In [19]:
SYSTEM_PROMPT = """
Human: 你是一个专业的法律 AI 助手，擅长解答《中华人民共和国民法典》相关的问题。你能够从提供的法律条文片段中找到准确的答案，并引用具体法条编号。
"""

for i, (q, ctx) in enumerate(zip(questions, all_contexts)):
    USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的法律条文片段来回答用 <question> 标签括起来的问题。
回答要求：
1. 准确引用相关法条编号
2. 用通俗易懂的语言解释法条含义
3. 如果检索到的法条不足以完整回答问题，请如实说明
<context>
{ctx}
</context>
<question>
{q}
</question>
"""

    print(f"\n{'='*60}")
    print(f"问题 {i+1}：{q}")
    print('='*60)

    response = deepseek_client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ],
    )
    print(response.choices[0].message.content)
    print()


问题 1：不动产物权未经登记是否发生效力？
根据《中华人民共和国民法典》第二百零九条的规定，**不动产物权未经登记，原则上不发生效力**，但法律另有规定的除外。

通俗解释：通常情况下，如果不动产物权（比如房产所有权）要设立、变更、转让或消灭，必须到不动产登记机构办理登记手续，登记完成后物权才具有法律效力。如果未办理登记，该物权变动对第三方没有法律约束力。

但是，法律也规定了两点例外：
1. **法律另有规定**：有时其他法律可能规定某些特殊情况不用登记也能生效（比如继承、法院判决等情形）。
2. **国家所有的自然资源**：例如国有土地、森林等自然资源，其所有权可以不登记。

此外，根据**第二百一十四条**，依法应当登记的，物权自记载于不动产登记簿时起生效。所以，**未登记通常不发生效力**，但需注意“法律另有规定”的例外情况。


问题 2：抵押权人能否未经抵押人同意转让抵押财产？
根据您提供的《中华人民共和国民法典》法律条文，**抵押权人不能未经抵押人同意转让抵押财产**，但存在例外情况。

具体解释如下：

1. **基本原则**：依据 **《中华人民共和国民法典》第四百一十六条** 的规定，抵押期间，**抵押人（即提供抵押财产的人）** 未经抵押权人（即债权人）同意，不得转让抵押财产。反之，此条文的逻辑也隐含了抵押权人作为债权人，同样不能随意处置抵押物，因为抵押财产的所有权仍属于抵押人。

2. **例外情形**：上述条文明确了一个例外——如果**受让人（即第三方）** 同意代为清偿债务，或者为抵押权人提供其他担保，那么转让可以被允许。也就是说，只有在债务问题得到妥善解决或担保得到替换的情况下，转让才可能有效。

3. **清晰说明**：需要区分的是，您提供的法条主要规范的是“抵押人”的行为（即抵押人未经抵押权人同意不得转让），而问题问的是“抵押权人”能否转让。但从法律精神和体系解释来看，抵押权人享有的是对抵押物的优先受偿权，而非所有权，因此**无权自行决定转让抵押财产**。如果抵押权人想实现抵押权，应当依据 **第四百一十九条** 的规定，通过与抵押人协议折价、拍卖或变卖等方式进行，或在协议不成时请求法院拍卖、变卖。

**通俗总结**：抵押权人不能直接把抵押物卖给别人，因为东西是抵押人的。只有在有人愿意替还债或提供新担保时，才可能转让。否则，抵押

In [21]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的法律条文片段来回答用 <question> 标签括起来的问题。\n回答要求：\n1. 准确引用相关法条编号\n2. 用通俗易懂的语言解释法条含义\n3. 如果检索到的法条不足以完整回答问题，请如实说明\n<context>\n**第四百一十六条** 抵押期间，抵押人未经抵押权人同意，不得转让抵押财产，但是受让人代为清偿债务或者向抵押权人提供担保的除外。\n抵押人转让抵押财产的，转让所得的价款，应当向抵押权人提前清偿债务或者提存。\n**第四百四十条** 质权期间，质权人未经出质人同意，不得转让质押财产，但是受让人代为清偿债务或者向质权人提供担保的除外。\n质权人转让质押财产的，转让所得的价款，应当向质权人提前清偿债务或者提存。\n**第四百一十九条** 抵押权人实现抵押权，可以与抵押人协议以抵押财产折价或者以拍卖、变卖该抵押财产所得的价款优先受偿。\n协议损害其他债权人利益的，其他债权人可以请求人民法院撤销该协议。\n抵押权人与抵押人未就抵押权的实现方式达成协议的，抵押权人可以请求人民法院拍卖、变卖抵押财产。\n</context>\n<question>\n抵押权人能否未经抵押人同意转让抵押财产？\n</question>\n'

使用 DeepSeek 提供的 `deepseek-chat` 模型对 2 个问题逐一生成 RAG 响应。

In [22]:
# 上一个 cell 中已完成所有问题的 RAG 检索与回答